In [1]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
import joblib
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('../data/train.csv')

In [3]:
X = df.drop(['Churn','CustomerID'], axis=1)
y = df['Churn']

X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [4]:
cat_cols = ['SubscriptionType', 'PaymentMethod', 'ContentType', 'GenrePreference', 'DeviceRegistered', 'Gender', 'PaperlessBilling', 'MultiDeviceAccess', 'ParentalControl', "SubtitlesEnabled"]

encoders = joblib.load('../models/encoders.joblib')

for col in cat_cols:
    X_test_rf[col] = encoders[col].transform(X_test_rf[col]).astype(str)

In [5]:
# Convert all your string/object columns first
categorical_cols = X_test_xgb.select_dtypes(include=['object']).columns
for col in categorical_cols:
    X_test_xgb[col] = X_test_xgb[col].astype('category')

C:\Users\kumar\AppData\Local\Temp\ipykernel_11248\1630224740.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_test_xgb.select_dtypes(include=['object']).columns


In [6]:
rf = joblib.load('../models/randomForest.joblib')
xgb = joblib.load('../models/xgboost.joblib')

In [7]:
def evaluate_model(name, y_true, y_pred, y_prob, threshold):
    return {
        'Model': name,
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Churn Precision': precision_score(y_true, y_pred),
        'Churn Recall': recall_score(y_true, y_pred),
        'Churn F1': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }

In [8]:
X_test_rf

,AccountAge,MonthlyCharges,TotalCharges,SubscriptionType,PaymentMethod,PaperlessBilling,ContentType,MultiDeviceAccess,DeviceRegistered,ViewingHoursPerWeek,AverageViewingDuration,ContentDownloadsPerMonth,GenrePreference,UserRating,SupportTicketsPerMonth,Gender,WatchlistSize,ParentalControl,SubtitlesEnabled
121976,82,15.211698,1247.359265,1,1,0,0,0,0,21.365122,10.905371,16,2,1.564454,3,0,5,1,0
232925,47,12.490732,587.064385,1,3,1,1,1,3,3.241988,5.658260,40,1,2.891674,5,0,13,0,1
61236,113,13.057703,1475.520435,2,3,1,2,0,1,23.038332,127.349847,7,2,1.788752,8,1,22,0,1
222253,31,9.669522,299.755172,1,0,0,0,0,0,2.939787,15.027302,48,1,3.665060,4,1,10,0,0
176454,34,18.814475,639.692163,2,2,1,1,1,3,1.998127,141.789040,28,0,2.677158,6,0,10,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110079,47,12.565654,590.585744,2,1,0,0,1,1,16.417081,165.201421,31,1,1.398329,9,0,12,1,0
145223,21,14.579965,306.179262,1,3,0,1,0,3,30.349524,33.598322,19,1,4.290966,5,0,0,1,1
159916,72,18.480613,1330.604144,1,3,1,1,1,0,34.445074,42.207345,42,3,4.369914,8,1,17,1,0
229249,33,5.514994,181.994787,1,3,1,1,1,0,11.466072,157.819000,0,4,3.717602,0,1,13,1,1


In [9]:
print(xgb.feature_names_in_)

['AccountAge' 'MonthlyCharges' 'TotalCharges' 'SubscriptionType'
 'PaymentMethod' 'PaperlessBilling' 'ContentType' 'MultiDeviceAccess'
 'DeviceRegistered' 'ViewingHoursPerWeek' 'AverageViewingDuration'
 'ContentDownloadsPerMonth' 'GenrePreference' 'UserRating'
 'SupportTicketsPerMonth' 'Gender' 'WatchlistSize' 'ParentalControl'
 'SubtitlesEnabled']


In [11]:
# xgboost
y_prob_xgb = xgb.predict_proba(X_test_xgb)[:, 1]
y_pred_xgb = (y_prob_xgb >= 0.55).astype(int)

# random forest
y_prob_rf = rf.predict_proba(X_test_rf)[:, list(rf.classes_).index(1)]
y_pred_rf = (y_prob_rf >=0.35).astype(int)

In [12]:
rf_results = evaluate_model(
    'Random Forest',
    y_test_rf,
    y_pred_rf,
    y_prob_rf,
    0.35
)

xgb_results = evaluate_model(
    'XGBoost',
    y_test_xgb,
    y_pred_xgb,
    y_prob_xgb,
    0.55
)

comparison = pd.DataFrame([
    rf_results,
    xgb_results
])

comparison

,Model,Threshold,Accuracy,Churn Precision,Churn Recall,Churn F1,ROC-AUC
0,Random Forest,0.35,0.700644,0.328713,0.625325,0.430911,0.738274
1,XGBoost,0.55,0.722097,0.347683,0.608691,0.442570,0.750386
